# Lab 09: How to answer follow-up questions with retrieval

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Answer a follow-up question by searching with a query that stands on its own, and check that query in code before you
trust it. The moment that teaches it: the user asks how to reset a Boreal password, then *How long is the link
valid?* Searched as typed, the question finds the Atlas policy first, and the answer gives Atlas's reset-link time with
a citation whose quote really is in the evidence, so a quote check passes. One chat call that rewrites
the question to name Boreal fixes it, and two short checks in code say whether a rewrite kept the product and added
no number the user did not type.

Every question in the labs so far has stood on its own. In a
conversation the second question often names nothing (*How long is the link valid?*), and the search reads only the
words it is given, so this lab adds a step between the conversation and the search. The self-study parts let the model
write the query itself, as a search-tool call, and test both ways on seven conversations.

## What you will learn

- Why a follow-up searched as typed can find the wrong policy while the answer's citation passes a quote check.
- How to rewrite a follow-up into a question that stands on its own, with one chat call (query rewriting).
- How to check a rewritten query in code: the product kept, and no number the user did not type.
- How to let the model write the query itself as a search-tool call (`@tool` and `bind_tools`), and run the call.
- How to test both ways on a set of conversations over several runs, and see where an earlier answer leaks into a
  query.
- When to ask the user which product they mean instead of guessing, and why that rule belongs in code.

**In class:** Parts A to C, except the ten-run count at the end of Part A. That subsection and Parts D, E and F are
self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-27
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` from `models.env`. It embeds the
corpus once (about 21,000 tokens) and makes about 120 short chat calls, well under one US cent on `gpt-4o-mini`. The
saved outputs and every count in the text come from `gpt-4o-mini`; on another model the failures can move to other questions.
Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third
decimal, so compare the order of results and the kind of reply with the text, not every number.

A **follow-up** is a question that leans on the turns before it: *What about Boreal?* means nothing alone. Search
that has to work across the turns of a conversation is called **conversational** (or **multi-turn**) **retrieval**.
The pipeline from Labs 02 and 05 searches with one question. This lab adds the step that turns a conversation's last
turn into a question the search can use, and checks that say whether it kept what the user meant.

---

## Part A: Search a follow-up as typed

Part A runs the Lab 05 pipeline on a conversation's last turn, as it would on any single question, and shows what
goes wrong.

The conversations in this lab's `resources/conversations.json` are constructed: short help-desk exchanges about the
corpus's two products, Atlas and Boreal, with each earlier assistant turn written from one policy and citing it. Load
them, and look at the one this part follows:

In [3]:
# conversations: the seven conversations in the file, a list of dictionaries; by_id finds one by its id
import json

LAB = ROOT / "09_Follow_Up_Questions"
conversations = json.loads((LAB / "resources" / "conversations.json").read_text(encoding="utf-8"))["conversations"]
by_id = {conversation["id"]: conversation for conversation in conversations}

# case: the conversation Parts A to C follow
case = by_id["link_after_boreal"]
case

{'id': 'link_after_boreal',
 'slice': 'reference resolution',
 'turns': [{'role': 'user', 'content': 'How do I reset my Boreal password?'},
  {'role': 'assistant',
   'content': 'Boreal password reset uses a mobile app [H04].'},
  {'role': 'user', 'content': 'How long is the link valid?'}],
 'products': ['Boreal'],
 'source_id': 'H04'}

One conversation is a dictionary with five fields:

- `id` and `slice`: its name, and the kind of follow-up it tests (its **slice**); here the last turn refers back to
  something earlier (*the link*), so the slice is *reference resolution*.
- `turns`: the messages so far, oldest first, each a dictionary with a `role` (`user` or `assistant`) and its
  `content`.
- `products`: the product a search for the last turn must keep, here `['Boreal']`.
- `source_id`: the policy that answers the last turn, here `H04`.

`products` and `source_id` are an answer key: the checks in this lab read them, and the model never sees them.

The user asked how to reset a Boreal password, got an answer citing H04, and now asks *How long is the link valid?*,
meaning Boreal's reset link. List all seven conversations by their last turn:

In [4]:
# One line per conversation: id, slice, products, the answering policy, the last turn
for conversation in conversations:
    print(f"{conversation['id']:18}{conversation['slice']:22}{conversation['products']!s:20}"
          f"{conversation['source_id']!s:6}{conversation['turns'][-1]['content']}")

topic_switch      topic switch          ['Boreal']          H04   What about Boreal?
link_after_boreal reference resolution  ['Boreal']          H04   How long is the link valid?
account_type      reference resolution  ['Atlas']           H01   Can temporary accounts do that?
pronoun           reference resolution  ['Atlas']           H03   Who can export all of them?
earlier_answer    reference resolution  ['Atlas']           H05   Where can I download them?
evidence_gap      evidence gap          ['Boreal']          None  What about Boreal?
two_products      ambiguous             ['Atlas', 'Boreal'] None  How long is the link valid?


Five conversations have a policy that answers the last turn; `evidence_gap` has none (no policy covers Boreal's
billing), and in `two_products` the answer depends on which product the user means. Parts A to C follow
`link_after_boreal` and `topic_switch`; Part E tests all seven.

Build the vector store. Lab 06 Part D shows the pipeline in `shared/lc_pipeline.py` that Lab 05 rebuilt with
LangChain: `build_store()` embeds the documents into an in-memory vector store, `retrieve()` returns the three closest
documents with their cosine scores, `evidence()` writes them as `[ID] text` lines, and `answer()` makes one chat call
that returns a cited answer. `load_corpus()` and `to_documents()` read the 272 SQuAD chunks and the five policies and
wrap each one as a LangChain `Document`.

In [5]:
# store: the 277 documents (272 chunks and the five policies), embedded into a fresh InMemoryVectorStore
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
print(len(store.store), "documents in the store")

277 documents in the store


277 documents, embedded once for this run. Now search with the last turn exactly as the user
typed it:

> "How long is the link valid?"

*Predict first:* which policy will come first, Atlas's H02 or Boreal's H04?

`show()` prints each hit's score, ID and the start of its text:

In [6]:
# show() prints one line per hit; each hit is a (Document, cosine score) pair
def show(hits):
    for document, score in hits:
        print(f"  {score:.4f}  {document.id:13}{document.page_content[:70]}")

# follow_up: the last turn as typed; hits: the three closest documents for it
follow_up = case["turns"][-1]["content"]
hits = lc_pipeline.retrieve(store, follow_up)
show(hits)

  0.5949  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3770  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.1892  H01          Atlas passwords can be reset by email. Temporary accounts require a su


The Atlas policy H02 comes first (about 0.59) and Boreal's H04 second (about 0.38). **The search reads only the words
it is given**, and *the link* matches H02, which is all about reset links, better than H04, which also covers the
mobile app.

Each hit is a pair: a LangChain `Document` (`document.id` is the chunk or policy ID, `document.page_content` its text)
and its cosine score. Now answer the question with the pipeline's answer step. First the evidence the model will read:

In [7]:
# lines: the three hits as "[ID] text" lines, the evidence the answer step reads
lines = lc_pipeline.evidence(hits)
for line in lines:
    print(line)

[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[H04] Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.


Three lines, best hit first, each starting with the ID a citation can point to. Make one answer call with the
question as typed and these lines:

In [8]:
# reply: one cited answer for the follow-up as typed
reply = lc_pipeline.answer(follow_up, lines, settings)
reply

CitedAnswer(answer='Atlas reset links expire after 30 minutes.', claims=[Claim(claim='Atlas reset links expire after 30 minutes.', source_id='H02', quote='Atlas reset links expire after 30 minutes.')])

`reply` is a `CitedAnswer`, the Pydantic object `answer()` returns (Lab 06 Part D prints its source):

- `reply.answer`: the sentence for the user;
- `reply.claims`: a list of `Claim` objects, each with the `claim`, the `source_id` it cites and a `quote` copied from
  that evidence line.

This reply says *Atlas reset links expire after 30 minutes.*, and its one claim cites H02.

Now the quote check, `quote_found(claim, lines)` from `shared/answer.py`:

- it finds the evidence line whose ID is the claim's `source_id`, and returns `False` when no line has that ID;
- it compares words only, in lower case, so punctuation and capitals do not matter;
- it returns `True` when the quote's words appear in that line, in order (a quote shortened with `...` passes when
  each part is there).

It reads a dictionary, so `model_dump()` turns each `Claim` into one:

In [9]:
# For each claim: the cited ID, whether its quote is in that evidence line, and the quote
from shared.answer import quote_found

for claim in reply.claims:
    print(claim.source_id, quote_found(claim.model_dump(), lines), "|", claim.quote)

H02 True | Atlas reset links expire after 30 minutes.


`True`: the quote *Atlas reset links expire after 30 minutes.* is word for word in H02's line. **The citation is real
and the quote is in the evidence, so the check passes, for the wrong product:** the user asked about Boreal.

One reply can mislead, since replies vary from run to run. `answers()` asks the same question several times and groups
identical answers, so you see each different answer once, with how often it came back:

In [10]:
# answers() asks the same question `runs` times and counts each different answer, with its citations checked
from collections import Counter

def answers(question, hits, runs=3):
    lines = lc_pipeline.evidence(hits)
    seen = Counter()
    for _ in range(runs):
        reply = lc_pipeline.answer(question, lines, settings)
        checked = [(claim.source_id, quote_found(claim.model_dump(), lines)) for claim in reply.claims]
        seen[f"{reply.answer} | cited, quote found: {checked}"] += 1
    for text, count in seen.items():
        print(f"  {count} of {runs}: {text}")

### What happens step by step

1. `lc_pipeline.evidence(hits)` writes the hits as `[ID] text` lines, once.
2. The loop makes `runs` answer calls with the same question and the same lines.
3. `checked` pairs each claim's source ID with the quote check, as in the cell before.
4. `seen` is a `Counter`, a dictionary that counts how many times each key was added. The key is the answer and its
   checks as one line of text, so identical answers add up.
5. The last loop prints each different answer once, with its count.

Run it on the follow-up as typed, three times:

In [11]:
# Three answers to the follow-up as typed, from the same three hits
answers(follow_up, hits)

  2 of 3: Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes. | cited, quote found: [('H02', True), ('H04', True)]
  1 of 3: Atlas reset links expire after 30 minutes. | cited, quote found: [('H02', True)]


When we ran it, 1 of 3 answers said *Atlas reset links expire after 30 minutes.*, citing H02 with its quote found, and
2 of 3 gave Atlas's and Boreal's times together, citing H02 and H04 with both quotes found. None answered for Boreal
alone. **Every answer passes the quote check, and every one gives Atlas's 30 minutes to a user who asked about
Boreal;** the two-product answers add Boreal's 10 minutes and leave the user to pick. Nothing in the pipeline knew the
user meant Boreal.

Three runs is a small count, and the split moves from run to run. Over the runs of ten we made while building and
re-running the lab, Atlas's time alone came back 3 to 10 times out of ten; the rest gave both products' times, or now
and then *Insufficient evidence.* An answer for Boreal alone never came back. If none of your three answers gives Atlas alone, look at the ones that
give both times: they still hand Atlas's 30 minutes to a user who asked about Boreal.

The first conversation, `topic_switch`, fails another way. The user asked how long an Atlas reset link stays valid,
got *30 minutes*, then asked:

> "What about Boreal?"

*Predict first:* this turn names Boreal. Will the search find H04, and will the answer give Boreal's time?

In [12]:
# switch: the topic_switch conversation; its last turn searched as typed, then answered three times
switch = by_id["topic_switch"]
switch_hits = lc_pipeline.retrieve(store, switch["turns"][-1]["content"])
show(switch_hits)
answers(switch["turns"][-1]["content"], switch_hits)

  0.3256  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.3202  S16-P01-C01  rainforest spanned a relatively narrow band. It expanded again during 
  0.2958  S16-P01-C00  Following the Cretaceous–Paleogene extinction event, the extinction of


  3 of 3: Insufficient evidence. | cited, quote found: []


H04 comes first, but only just: about 0.33, against about 0.32 for an unrelated chunk about a rainforest. *Boreal* is
the only word in the turn the search can use. The answer refuses 3 of 3, with the right policy first in its evidence,
because *What about Boreal?* asks nothing: the task (how long a reset link stays valid) is in the turn before.

So the last turn alone can lose the product (the search finds Atlas) or the task (the answer has no question). Both
come from one gap: the pipeline sees one turn, and the meaning is spread over three.

**Lesson to Remember**
> A follow-up searched as typed can find the wrong policy, and the answer can cite it with a quote that really is in
> the evidence. A citation check tests the answer against its evidence; it cannot tell that the evidence is about the
> wrong product.

### Checkpoint A

| You should see | Our run |
|---|---|
| Conversations | 7, from `topic_switch` to `two_products` |
| *How long is the link valid?* as typed | H02 0.5949, H04 0.3770, H01 0.1892 |
| One answer, quote check | *Atlas reset links expire after 30 minutes.*, cites H02, quote found `True` |
| Three answers | Atlas's 30 minutes alone 1 of 3; both products' times 2 of 3; every quote found |
| *What about Boreal?* as typed | H04 0.3256, then S16-P01-C01 0.3202 (*rainforest spanned ...*); *Insufficient evidence.* 3 of 3 |

### Count the as-typed answers over ten runs (self-study)

Three runs is a small count. Ask the same follow-up ten times, with the same three hits:

In [13]:
# Ten answers to the follow-up as typed, to see the spread three runs can hide
answers(follow_up, hits, runs=10)

  2 of 10: Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes. | cited, quote found: [('H02', True), ('H04', True)]
  8 of 10: Atlas reset links expire after 30 minutes. | cited, quote found: [('H02', True)]


When we ran it, 8 of 10 answers gave Atlas's 30 minutes alone and 2 of 10 gave both products' times; none answered for
Boreal alone. Over our runs of ten, Atlas alone came back 3 to 10 times. **The kind of answer holds from run to run;
the split does not,** so report a count with its number of runs.

| You should see | Our run |
|---|---|
| Ten answers as typed | Atlas's 30 minutes alone 8 of 10; both products' times 2 of 10 |

---

## Part B: Rewrite the follow-up as a standalone question

Part B adds one chat call before the search. It rewrites the conversation's last turn as a **standalone question**: a
question that means what the last turn means and can be understood without the turns before it. Adding this step is
called **query rewriting**, and it is the usual first fix for follow-ups in conversational retrieval.

The model needs the conversation as text. Write each turn as `role: content`, one per line:

In [14]:
# history_text() writes the turns as one string, one "role: content" line per turn
def history_text(turns):
    return "\n".join(f"{turn['role']}: {turn['content']}" for turn in turns)

print(history_text(case["turns"]))

user: How do I reset my Boreal password?
assistant: Boreal password reset uses a mobile app [H04].
user: How long is the link valid?


Three lines of plain text, the way a person would read the conversation.

The instruction says what to keep and what not to do. A rewrite that answers the question (*Boreal links expire after
10 minutes*) would search for its own guess, and a guess that matches a policy's words would look supported. Build the
two messages:

In [15]:
# REWRITE: the system instruction; messages: the instruction, then the whole conversation as one user message
REWRITE = ("Rewrite the last user turn as a standalone search query. "
           "Keep products, account types and conditions. "
           "Do not answer it. Return only the query.")
messages = [("system", REWRITE), ("user", history_text(case["turns"]))]
messages

[('system',
  'Rewrite the last user turn as a standalone search query. Keep products, account types and conditions. Do not answer it. Return only the query.'),
 ('user',
  'user: How do I reset my Boreal password?\nassistant: Boreal password reset uses a mobile app [H04].\nuser: How long is the link valid?')]

A list of two messages, each a `(role, text)` pair. LangChain accepts this short form for a message as well as the
`{"role": ..., "content": ...}` dictionaries the conversation file uses. The first message is the instruction
(`system`); the second holds the three turns as one `user` message, so the model reads the conversation as text to
rewrite, not as a chat to continue.

`lc_chat_model()` from `shared/settings.py` builds the LangChain chat model that `models.env` names (Lab 07 built its judge with it).
`invoke()` sends the messages and returns the model's message; `.content` is its text. This is one plain call, with no
answer schema:

In [16]:
# chat: the chat model from models.env; query: the rewritten question, the text of the reply
from shared.settings import lc_chat_model

chat = lc_chat_model("CHAT_MODEL", settings)
query = chat.invoke(messages).content
print(query)

Boreal password reset link validity duration.


The rewrite is *Boreal password reset link validity duration.*: not a full sentence, and it does not need to be, since
it names the product and the task. Search and answer with it, as in Part A:

In [17]:
# query_hits: the three closest documents for the rewritten query; then three answers to it
query_hits = lc_pipeline.retrieve(store, query)
show(query_hits)
answers(query, query_hits)

  0.7825  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.5801  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.5124  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: Boreal reset links expire after 10 minutes. | cited, quote found: [('H04', True)]


H04 now comes first (about 0.78, with H02 at about 0.58), and all three answers say *Boreal reset links expire after
10 minutes.*, citing H04 with the quote found. **The rewrite moved the meaning from the earlier turns into the query,
and the search and the answer both got it.**

The call is the same for every conversation, so put it in a function, and try it on *What about Boreal?*, which Part A
could not answer:

In [18]:
# rewrite() makes Part B's call for any conversation's turns and returns the query
def rewrite(turns):
    return chat.invoke([("system", REWRITE), ("user", history_text(turns))]).content

# switch_query: the rewrite of topic_switch
switch_query = rewrite(switch["turns"])
print(switch_query)

How long does a Boreal reset link remain valid?


*How long does a Boreal reset link remain valid?*: the task from the first, Atlas question, and the product from the
last turn. Search and answer with it:

In [19]:
# The rewritten topic_switch query: its three closest documents, then three answers
switch_hits = lc_pipeline.retrieve(store, switch_query)
show(switch_hits)
answers(switch_query, switch_hits)

  0.6734  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.6418  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3650  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: Boreal reset links remain valid for 10 minutes. | cited, quote found: [('H04', True)]


H04 leads at about 0.67, with H02 close behind at about 0.64, and the three answers give Boreal's 10 minutes, citing
H04, where Part A refused.

**Lesson to Remember**
> The search sees only the query, and a follow-up means little without the turns before it. Query rewriting moves the
> meaning of the earlier turns into one standalone question before the search, so the search and the answer both get
> it.

### Checkpoint B

| You should see | Our run |
|---|---|
| Rewrite of *How long is the link valid?* | *Boreal password reset link validity duration.* |
| Its search and answers | H04 0.7825, H02 0.5801; *Boreal reset links expire after 10 minutes.* 3 of 3, quote found |
| Rewrite of *What about Boreal?* | *How long does a Boreal reset link remain valid?*; H04 0.6734, H02 0.6418; Boreal's 10 minutes 3 of 3 |

---

## Part C: Check the rewrite in code

Part C writes two checks that read a query and say whether it kept what the user meant. A rewrite is model output and
can drop or add words on any run; a check in code costs nothing and runs on every query.

- **Keep the product.** The conversation file names the product each follow-up is about; the check passes when each
  one appears in the query.
- **Add no answer.** A query that answers, or copies an earlier answer, brings in numbers the user never typed (*30
  minutes*, a source ID such as *H02*). The check lists every number in the query that is not in a user turn.

Start with the product check, on the rewrite from Part B:

In [20]:
# products_kept() is True when every product in the answer key appears in the query, upper or lower case
def products_kept(query, case):
    return all(product.lower() in query.lower() for product in case["products"])

print(f"{query!r}: product kept {products_kept(query, case)}")

'Boreal password reset link validity duration.': product kept True


`all()` is `True` only when the test holds for every item, here every product in `case["products"]`; `.lower()`
makes *boreal* and *Boreal* match.

Now the number check. `re.findall(r"\d+", text)` returns every run of digits in the text as a list of strings: `\d`
is one digit, and `+` means one or more.

In [21]:
# new_numbers() lists the numbers in the query that none of the user's turns contains
import re

def new_numbers(query, turns):
    typed = re.findall(r"\d+", " ".join(turn["content"] for turn in turns if turn["role"] == "user"))
    return [number for number in re.findall(r"\d+", query) if number not in typed]

print(f"{query!r}: new numbers {new_numbers(query, case['turns'])}")

'Boreal password reset link validity duration.': new numbers []


`typed` holds every number in the user's turns, and the list keeps each number in the query that is not among them.
The assistant's turns are left out on purpose: a number the assistant said, copied into the query, is the leak this
check is for. Part B's rewrite has no number, so the list is empty, and the rewrite passes both checks.

A check is only useful if it can fail, so try two queries written by hand: the follow-up as typed, and a rewrite of
*What about Boreal?* that carries the Atlas answer into the search:

In [22]:
# Two bad queries written by hand, each checked against its own conversation
for bad, bad_case in (("How long is the link valid?", case),
                      ("Do Boreal reset links also expire after 30 minutes?", switch)):
    print(f"{bad!r}: product kept {products_kept(bad, bad_case)} | new numbers {new_numbers(bad, bad_case['turns'])}")

'How long is the link valid?': product kept False | new numbers []
'Do Boreal reset links also expire after 30 minutes?': product kept True | new numbers ['30']


The first fails the product check; the second keeps Boreal and fails the number check with `['30']`, the earlier
Atlas answer. **Neither check reads meaning**: a query could name Boreal and still ask the wrong question. They catch
the two failures this lab is about, on every query, without a model call. Part E (self-study) runs both checks on 84
queries written four ways.

**Lesson to Remember**
> A rewrite is model output, so check it like model output: in code, on every query, before the search uses it. Try a
> check on a bad case before you trust its passes.

### Checkpoint C

| You should see | Our run |
|---|---|
| Part B's rewrite | product kept `True`; new numbers `[]` |
| The two hand-written queries | `False`, `[]`; then `True`, `['30']` |

---

## Part D: Let the model write the query as a tool call (self-study)

Part D gives the model a search tool and lets it write the query itself, in the same call that reads the
conversation. This is the route an agent takes: retrieval as a tool the model may call, often called **agentic RAG**.

A **tool** is a function the model may ask you to run. LangChain's
[`@tool`](https://docs.langchain.com/oss/python/langchain/tools) decorator turns a Python function into one, and the
docstring becomes the description the model reads. `convert_to_openai_tool()` shows a tool in the form the chat API
receives. Make the search a tool, and print what the model is sent:

In [23]:
# search_documents: Lab 05's search as a LangChain tool; its docstring is the description the model reads
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def search_documents(query: str) -> str:
    """Search the help-centre documents and return the three closest passages. The query must make sense on its own."""
    return "\n".join(lc_pipeline.evidence(lc_pipeline.retrieve(store, query)))

# What the model is sent for this tool
print(convert_to_openai_tool(search_documents))

{'type': 'function', 'function': {'name': 'search_documents', 'description': 'Search the help-centre documents and return the three closest passages. The query must make sense on its own.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}


The model never sees your code, only this dictionary:

- `name`: `search_documents`, the function's name;
- `description`: the docstring, word for word;
- `parameters`: a JSON schema with one string argument, `query`, marked required.

`bind_tools()` returns a copy of the chat model that sends this tool with every call. This time the conversation goes
in as messages, not as text to rewrite: a system message that says to search first, then the three turns, which
`*case["turns"]` spreads into the list as they are in the file.

> "How long is the link valid?" (after a question about Boreal)

*Predict first:* the model reads all three turns. Will the query it writes name Boreal?

In [24]:
# SYSTEM: the rules for the tool route; with_search: the chat model with the tool attached
SYSTEM = ("You answer questions about the Atlas and Boreal products. Search the documents before you answer."
          " Answer only from the passages, citing their IDs.")
with_search = chat.bind_tools([search_documents])

# tool_messages: the system message, then the three turns; reply: the model's first reply
tool_messages = [{"role": "system", "content": SYSTEM}, *case["turns"]]
reply = with_search.invoke(tool_messages)
print("text:", repr(reply.content))
print("tool calls:", reply.tool_calls)

text: ''
tool calls: [{'name': 'search_documents', 'args': {'query': 'link validity'}, 'id': 'call_UDCNswiERmp67OgV5n9o7fMZ', 'type': 'tool_call'}]


The reply has no text (`''`). `reply.tool_calls` is a list with one **tool call**, a dictionary with three keys:

- `name`: the tool to run;
- `args`: the arguments the model wrote, here `query`;
- `id`: the call's ID, which the result must carry back.

Check the query the model wrote with Part C's checks:

In [25]:
# tool_query: the query in the model's tool call, checked like a rewrite
tool_query = reply.tool_calls[0]["args"]["query"]
print(f"{tool_query!r}: product kept {products_kept(tool_query, case)} | new numbers {new_numbers(tool_query, case['turns'])}")

'link validity': product kept False | new numbers []


*link validity*: the product is gone, and the check says so. Nothing has searched yet: the model asked, and your code runs the tool. `search_documents.invoke()` with the
whole tool call runs the function on its `args` and returns a `ToolMessage`: a message that carries the result and the
call's ID, so the model can match the result to its request.

In [26]:
# tool_message: the search result for the model's call, as a ToolMessage with the call's ID
tool_message = search_documents.invoke(reply.tool_calls[0])
print(type(tool_message).__name__, "| same ID as the call:", tool_message.tool_call_id == reply.tool_calls[0]["id"])
print(tool_message.content)

ToolMessage | same ID as the call: True
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[S19-P02-C01] leaky bucket. In case of a shared physical medium (such as radio or 10BASE5), the packets may be delivered according to a multiple access scheme.
[S19-P00-C01] the development of telecommunications in the Bell System. The new concept found little resonance among network implementers until the independent work of Donald Davies at the National Physical Laboratory (United Kingdom) (NPL) in the late 1960s. Davies is credited with coining the modern name packet switching and inspiring numerous packet switching networks in Europe in the decade following, including the incorporation of the concept in the early ARPANET in the United States.


The search found H02 and two chunks about packet switching (*leaky bucket*), and no Boreal policy at all. Send the conversation back with the model's tool call and the result, and the model writes its answer:

In [27]:
# final: the model's next reply, after reading its own tool call and the search result
final = with_search.invoke(tool_messages + [reply, tool_message])
print("tool calls:", final.tool_calls)
print(final.content)

tool calls: []
The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].


*The reset links for Atlas expire after 30 minutes ... [H02].* The model had the whole conversation in front of it,
and still answered from what its own query found.

A reply can ask for another search instead of answering, so the round trip goes in a loop: run every tool call, send
the results back, stop when a reply has none. `tool_answer()` does that and returns the model's queries with its final
answer:

In [28]:
# tool_answer() runs the tool route to the end and returns (the queries the model wrote, its final answer)
def tool_answer(turns, model):
    messages = [{"role": "system", "content": SYSTEM}, *turns]
    reply = model.invoke(messages)
    queries = []
    while reply.tool_calls:
        messages.append(reply)
        for call in reply.tool_calls:
            queries.append(call["args"]["query"])
            messages.append(search_documents.invoke(call))
        reply = model.invoke(messages)
    return queries, reply.content

### What happens step by step

1. `messages` starts as the system message and the turns, as in the first tool call above.
2. The first call may answer or ask for tools; `while reply.tool_calls` keeps going as long as the reply asks.
3. Inside the loop the model's reply goes on the list first, then one `ToolMessage` per call: the API rejects a tool
   call sent back without its result.
4. Each call's query is kept in `queries`, so you can check what the model searched for.
5. The next call reads everything so far. When its reply has no tool calls, the loop ends and its text is the answer.

Run it three times on the same conversation:

In [29]:
# The tool route, three times on link_after_boreal
for _ in range(3):
    print(tool_answer(case["turns"], with_search))

(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


The same query, *link validity*, and the same Atlas answer, 3 of 3.

The tool's description is the only instruction the model reads about the query, and *make sense on its own* did not
say what that takes here. Give the same function a description that names the product and the task, and run the loop
again; the search is unchanged. `tool("search_documents", description=...)` builds a decorator with that name and
description; applied to `search_documents.func`, the plain Python function inside the first tool, it makes a second
tool that runs the same search under the same name, so `tool_answer()` works unchanged:

In [30]:
# search_named: the same search with a description that names the product; with_named_search: the model with it
PRODUCT_RULE = ("Search the help-centre documents and return the three closest passages. Write a query that makes"
                " sense on its own: name the product (Atlas or Boreal) and the task.")
search_named = tool("search_documents", description=PRODUCT_RULE)(search_documents.func)
with_named_search = chat.bind_tools([search_named])

# The tool route with the new description, three times
for _ in range(3):
    print(tool_answer(case["turns"], with_named_search))

(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


The query is now *Boreal password reset link validity*, and the answer *The Boreal password reset links expire after
10 minutes [H04].*, 3 of 3.

The two routes put the same step in different places. The rewrite is a call your code makes before every search, so
you can check its query before anything uses it. The tool call lets the model decide when to search and what for,
which helps when a question needs several searches; its query follows the tool's description, not your rewrite
instruction. The bonus lab builds a full agent on this loop.

**Lesson to Remember**
> When the model writes its own search query, the tool's description is the instruction for that query. Check the
> query it wrote the same way you check a rewrite.

### Checkpoint D

| You should see | Our run |
|---|---|
| The tool as sent | `search_documents`, the description, one `query` string |
| First reply | no text, one tool call: *link validity*, product kept `False` |
| The loop, first description | *link validity* and Atlas's 30 minutes with H02, 3 of 3 |
| The loop, product rule | *Boreal password reset link validity* and Boreal's 10 minutes with H04, 3 of 3 |

---

## Part E: Test both routes on the seven conversations (self-study)

Part E runs every way of writing the query on every conversation, three times each, so that one run cannot decide.

Four ways: the last turn **as typed**, the **rewrite**, the **tool call** with the first description, and the tool
call with the **product rule**. For the tool routes only the query matters here, so `tool_query()` makes the first
call and reads its query without searching. A reply with no tool call gives an empty query, which fails the product
check:

In [31]:
# tool_query(): the query in the model's first tool call, or "" when it called no tool
def tool_query(turns, model):
    calls = model.invoke([{"role": "system", "content": SYSTEM}, *turns]).tool_calls
    return calls[0]["args"]["query"] if calls else ""

print(repr(tool_query(case["turns"], with_search)))

'link validity'


*link validity*, as in Part D: the query in the first tool call, with no search run.

Each route is a function from a conversation's turns to a query. `lambda turns: ...` writes a one-line function in
place: `lambda turns: turns[-1]["content"]` returns the last turn as typed. Put the four routes in a dictionary:

In [32]:
# routes: the four ways to write a query, each a function of the turns
routes = {"as typed": lambda turns: turns[-1]["content"],
          "rewrite": rewrite,
          "tool call": lambda turns: tool_query(turns, with_search),
          "product rule": lambda turns: tool_query(turns, with_named_search)}
list(routes)

['as typed', 'rewrite', 'tool call', 'product rule']

Now write three queries per conversation and route, and print each different query with how often it came back:

In [33]:
# queries: for each (conversation id, route), a Counter of the three queries that route wrote
queries = {}
for conversation in conversations:
    for route, write in routes.items():
        queries[(conversation["id"], route)] = Counter(write(conversation["turns"]) for _ in range(3))

# One line per conversation and route
for (case_id, route), counts in queries.items():
    print(f"{case_id:18}{route:14}" + " | ".join(f"{n}x {q!r}" for q, n in counts.items()))

topic_switch      as typed      3x 'What about Boreal?'
topic_switch      rewrite       3x 'How long does a Boreal reset link remain valid?'
topic_switch      tool call     3x 'Boreal reset link validity'
topic_switch      product rule  3x 'Boreal reset link validity'
link_after_boreal as typed      3x 'How long is the link valid?'
link_after_boreal rewrite       3x 'Boreal password reset link validity duration.'
link_after_boreal tool call     3x 'link validity'
link_after_boreal product rule  3x 'Boreal password reset link validity'
account_type      as typed      3x 'Can temporary accounts do that?'
account_type      rewrite       3x 'Can temporary accounts reset Atlas passwords by email?'
account_type      tool call     3x 'temporary accounts reset password email'
account_type      product rule  3x 'temporary accounts reset Atlas password by email'
pronoun           as typed      3x 'Who can export all of them?'
pronoun           rewrite       3x 'Who can export all reports in Atla

### What happens step by step

1. For each conversation and each route, `write(conversation["turns"])` runs three times. The rewrite and the two tool
   routes make a model call each time; as typed makes none.
2. `Counter(...)` counts identical queries, so three equal queries become one entry with the count 3.
3. `queries` has one entry per `(conversation id, route)` pair: 7 × 4 = 28 entries, 84 queries in all.
4. The print shows every different query with its count, `3x` when all three runs agreed.

28 lines, and most routes wrote the same query all three times. Look at the `rewrite` line for `earlier_answer`: two
of its three rewrites end in *[H05]*, the citation from the assistant's earlier turn.

Now count, per conversation and route, how many of the three queries keep the product, add no new number, and put the
answering policy first. The last count needs a policy, so it is blank for the two conversations with none. `first_id()`
searches with `k=1` and returns the ID of the closest document:

In [34]:
# first_id(): the ID of the closest document for a query (None for an empty query)
from shared.settings import chat_url

def first_id(query):
    return lc_pipeline.retrieve(store, query, k=1)[0][0].id if query else None

# One row per conversation and route: of the three queries, how many pass each test
print(f"{'conversation':18}{'route':14}{'product kept':>13}{'no new number':>15}{'right policy first':>20}")
for (case_id, route), counts in queries.items():
    conversation = by_id[case_id]
    kept = sum(n for q, n in counts.items() if products_kept(q, conversation))
    clean = sum(n for q, n in counts.items() if not new_numbers(q, conversation["turns"]))
    right = sum(n for q, n in counts.items() if first_id(q) == conversation["source_id"])
    print(f"{case_id:18}{route:14}{kept:>8} of 3{clean:>10} of 3"
          f"{(f'{right} of 3' if conversation['source_id'] else '-'):>20}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

conversation      route          product kept  no new number  right policy first


topic_switch      as typed             3 of 3         3 of 3              3 of 3


topic_switch      rewrite              3 of 3         3 of 3              3 of 3


topic_switch      tool call            3 of 3         3 of 3              3 of 3


topic_switch      product rule         3 of 3         3 of 3              3 of 3


...

evidence_gap      product rule         3 of 3         3 of 3                   -


two_products      as typed             0 of 3         3 of 3                   -


two_products      rewrite              3 of 3         3 of 3                   -


two_products      tool call            0 of 3         3 of 3                   -


two_products      product rule         3 of 3         3 of 3                   -

gpt-4o-mini at https://api.openai.com/v1


### What happens step by step

1. `counts` holds each different query a route wrote and how many of the three runs wrote it.
2. `kept`, `clean` and `right` add up the counts of the queries that pass each test, so each is a number from 0 to 3.
3. `first_id(q)` searches once per different query (one embedding call), not once per run.
4. The last line names the model and the endpoint behind the table.

Route by route:

- **As typed** fails the product check on five conversations, yet puts the right policy first on three of them: only
  Atlas has temporary accounts, report exports and invoices here, so the topic was enough. With a third product it
  would not be. `link_after_boreal` gets the wrong policy, 0 of 3.
- **The rewrite** keeps the product in all 21 queries and puts the right policy first in all 15 that have one. It
  fails the number check only on `earlier_answer`, 2 of 3.
- **The tool call** drops the product on `link_after_boreal`, `account_type` and `two_products`, 3 of 3 each, and puts
  the wrong policy first on `link_after_boreal`.
- **The product rule** passes every column.

The `earlier_answer` rewrites show an earlier answer leaking in: *Atlas invoice download location for monthly
invoices [H05].* (2 of 3) takes the assistant's *monthly* and its citation into the search. The number check catches
the *05*; the third rewrite, *... for monthly invoices.*, passes, because *monthly* is a word, not a number. **A check
sees only what it tests.**

While we built the lab, one `topic_switch` rewrite in three also answered instead of rewriting (*Boreal reset links
expire after 30 minutes [H02].*), with the wrong policy first; the number check caught it. In this run all three
`topic_switch` rewrites were clean.

Does an earlier answer reach a reply? In `evidence_gap` the user learned that Atlas invoices are sent *monthly*, then
asks *What about Boreal?* No policy covers Boreal's billing, so the right answer is a refusal; *monthly* for Boreal
would be the earlier answer reused. Run the rewrite route first:

In [35]:
# gap: the evidence_gap conversation; the rewrite route: its query, its hits, three answers
gap = by_id["evidence_gap"]
gap_query = rewrite(gap["turns"])
print(gap_query)
gap_hits = lc_pipeline.retrieve(store, gap_query)
show(gap_hits)
answers(gap_query, gap_hits)

How often are Boreal invoices sent?


  0.5380  H05          Atlas invoices are sent monthly. Workspace owners can download invoice
  0.3801  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.2491  H03          Atlas exports reports as CSV files. Only workspace owners can export a


  3 of 3: Insufficient evidence. | cited, quote found: []


The rewrite asks *How often are Boreal invoices sent?*, the search puts Atlas's billing policy H05 first (about 0.54),
and the answer refuses 3 of 3. Now the tool route with the product rule:

In [36]:
# The tool route with the product rule on evidence_gap, three times
for _ in range(3):
    print(tool_answer(gap["turns"], with_named_search))

(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only information about Atlas invoices being sent monthly is available [H05].')


(['Boreal invoices frequency'], 'The documents do not specify the frequency of Boreal invoices. Only the frequency for Atlas invoices is mentioned, which are sent monthly [H05].')


(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only information about Atlas invoices being sent monthly is available [H05].')


The tool route says 3 of 3 that the documents do not give Boreal's billing, and adds that Atlas invoices are sent
monthly, citing H05. *Monthly* reached the reply as Atlas's fact; no answer reused it for Boreal.

**Lesson to Remember**
> One run of one route proves little. A table of routes by conversations, over several runs, shows which failure
> belongs to which route, and the queries themselves show what leaked in from the earlier turns.

### Checkpoint E

| You should see | Our run |
|---|---|
| As typed | product kept 0 of 3 on five conversations; wrong policy first on `link_after_boreal` only |
| Rewrite | product kept 21 of 21; right policy first 15 of 15; *[H05]* in 2 of 3 `earlier_answer` queries |
| Tool call | product dropped on three conversations; wrong policy first on `link_after_boreal` |
| Product rule | 3 of 3 in every column |
| `evidence_gap` | rewrite: *Insufficient evidence.* 3 of 3; tool: no Boreal billing, Atlas's monthly cited to H05, 3 of 3 |

---

## Part F: Ask which product instead of guessing (self-study)

Part F handles the follow-up no query can resolve: the conversation does not say which product the user means.

In `two_products` the user asked about their Atlas password, then about a colleague's Boreal account, then *How long
is the link valid?* Which link? In Part E the query as typed and the first tool description named no product, so the
search put Atlas first, as in Parts A and D; the rewrite and the product rule named both. Each route chose a reading
without saying so. The honest reply is a question: *Which product do you mean, Atlas or Boreal?*

The first idea is to tell the rewrite step to ask. Add one line to the instruction and run it on all seven
conversations, where only `two_products` should ask:

In [37]:
# ASK: one more line for the rewrite instruction; each conversation is rewritten once with it
ASK = (" If the conversation does not make clear which product the last turn is about,"
       " return only: Which product do you mean, Atlas or Boreal?")
for conversation in conversations:
    text = chat.invoke([("system", REWRITE + ASK), ("user", history_text(conversation["turns"]))]).content
    print(f"{conversation['id']:18}{text}")

topic_switch      Which product do you mean, Atlas or Boreal?


link_after_boreal How long is the password reset link valid for Boreal accounts?


account_type      Can temporary accounts reset their Atlas password by email?


pronoun           Which product do you mean, Atlas or Boreal?


earlier_answer    Which product do you mean, Atlas or Boreal?


evidence_gap      Which product do you mean, Atlas or Boreal?


two_products      Which product do you mean, Atlas or Boreal?


It asked on five of the seven: `two_products`, and four that were clear, among them `topic_switch`, whose last turn
names Boreal. A rule the model applies with judgement fired where no judgement was needed.

This rule needs no judgement. Ask when the user has named more than one product and the last turn names none. Code can
test that exactly, on every turn:

In [38]:
# named() returns the products a text mentions; needs_question() applies the rule to the user's turns
PRODUCTS = ("Atlas", "Boreal")

def named(text):
    return {product for product in PRODUCTS if product.lower() in text.lower()}

def needs_question(turns):
    said = [turn["content"] for turn in turns if turn["role"] == "user"]
    return len(named(" ".join(said))) > 1 and not named(said[-1])

for conversation in conversations:
    print(f"{conversation['id']:18}{'ask which product' if needs_question(conversation['turns']) else 'rewrite and search'}")

topic_switch      rewrite and search
link_after_boreal rewrite and search
account_type      rewrite and search
pronoun           rewrite and search
earlier_answer    rewrite and search
evidence_gap      rewrite and search
two_products      ask which product


### What happens step by step

1. `named(text)` returns the set of products the text mentions, upper or lower case.
2. `said` holds the user's turns only, because the user's words say what they mean.
3. `needs_question()` is `True` when the user's turns together name more than one product and the last one names none.

Only `two_products` asks. A real help desk would also know the customer's product from their account, which beats any
rule (Lab 05's metadata filter).

Put the steps together: ask when the rule says so, otherwise rewrite, search and answer. Run it on Part A's
conversation and on `two_products`:

In [39]:
# answer_follow_up(): the question back when the rule says so, otherwise rewrite, search and answer
def answer_follow_up(turns):
    if needs_question(turns):
        return "Which product do you mean, Atlas or Boreal?"
    query = rewrite(turns)
    return lc_pipeline.answer(query, lc_pipeline.evidence(lc_pipeline.retrieve(store, query)), settings).answer

for case_id in ("link_after_boreal", "two_products"):
    print(f"{case_id:18}{answer_follow_up(by_id[case_id]['turns'])}")

link_after_boreal Boreal reset links expire after 10 minutes.
two_products      Which product do you mean, Atlas or Boreal?


*Boreal reset links expire after 10 minutes.* for the conversation that opened the lab, and a question back for the
one that cannot be answered without one.

**Lesson to Remember**
> A rule that can be written exactly belongs in code: it gives the same answer on every turn and costs no call. Leave
> to the model the steps that need reading.

### Checkpoint F

| You should see | Our run |
|---|---|
| The ask line in the instruction | asks on 5 of 7: `topic_switch`, `pronoun`, `earlier_answer`, `evidence_gap`, `two_products` |
| `needs_question()` | `ask which product` for `two_products` only |
| `answer_follow_up()` | Boreal's 10 minutes for `link_after_boreal`; *Which product do you mean, Atlas or Boreal?* for `two_products` |

---

## What you can do now

- Explain why a follow-up searched as typed can retrieve the wrong product, or leave the answer with no question,
  while every citation check passes.
- Rewrite a follow-up into a standalone query with one chat call, and check the query in code.
- Give the model a search tool with `@tool` and `bind_tools`, run its calls in a loop, and read the query it wrote.
- Test both routes on a set of conversations over several runs, and find where an earlier answer leaks in.
- Decide in code when to ask which product the user means, and say why a prompt line did not.

## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Add a conversation and test it on every route.** A user asks *Can I reset my Atlas password by email?*, gets
   *Yes. Atlas passwords can be reset by email [H01].*, and asks *Does Boreal work that way too?* H04 says Boreal's
   reset uses a mobile app, so the right answer is no.

   - Under **Task 1**, write the conversation as a dictionary shaped like those in `conversations.json` (`products`
     `["Boreal"]`, `source_id` `"H04"`). The cell answers it as typed, through the rewrite and through the tool with
     the product rule.

   **Check:** each route's answers. Our worked solution got *Yes, Boreal works that way too.* as typed (3 of 3, citing H04, every quote found);
*Insufficient evidence.* through the rewrite (3 of 3), whose query *Can I reset my Boreal password by email?* passed
both checks; and *No, Boreal does not work the same way. Password resets for Boreal use a mobile app ...* through the
tool (3 of 3).

   **Explain:** which route answered correctly, and why did the other two not? Would any check in this lab have
   flagged the answer as typed?

2. **Optional: check a rewrite instruction from an older tutorial.** LangChain once shipped query rewriting as a
   *history-aware retriever*, now in the legacy `langchain-classic` package; it was one chat call before the search,
   like Part B's. Tutorials for it often used: *Given a chat history and the latest user question which might
   reference context in the chat history, formulate a standalone question which can be understood without the chat
   history. Do NOT answer the question, just reformulate it if needed and otherwise return it as is.*

   - Under **Task 2**, set `OLD_REWRITE` to it. The cell rewrites each conversation three times and checks each query.

   **Check:** the queries that fail. Our worked solution got 6 of 21 failing, all on the product: *How long is the password reset link valid for?* for
`link_after_boreal` and `two_products`, 3 of 3 each.

   **Explain:** which part of this lab's `REWRITE` does the older instruction lack, and which conversation shows what
   it costs?

**Task 1.** Your conversation, then every route:

In [40]:
# Task 1: your conversation as a dictionary, then every route on it
new_case = None    # TODO: {"id": ..., "slice": ..., "turns": [...], "products": ["Boreal"], "source_id": "H04"}

if new_case is None:
    print("⚠️ Write the conversation, then run the cell again.")
else:
    last = new_case["turns"][-1]["content"]
    print("as typed:"); answers(last, lc_pipeline.retrieve(store, last))
    new_query = rewrite(new_case["turns"])
    print(f"rewrite: {new_query!r} | product kept {products_kept(new_query, new_case)}"
          f" | new numbers {new_numbers(new_query, new_case['turns'])}")
    answers(new_query, lc_pipeline.retrieve(store, new_query))
    print("tool, product rule:")
    for _ in range(3):
        print(" ", tool_answer(new_case["turns"], with_named_search))

⚠️ Write the conversation, then run the cell again.


**Task 2 (optional).** The older instruction, then the checks:

In [41]:
# Task 2: the older instruction, then three rewrites per conversation, each checked
OLD_REWRITE = None    # TODO: the instruction from the task, as a string

if OLD_REWRITE is None:
    print("⚠️ Set OLD_REWRITE, then run the cell again.")
else:
    for case in conversations:
        for _ in range(3):
            old = chat.invoke([("system", OLD_REWRITE), ("user", history_text(case["turns"]))]).content
            print(f"{case['id']:18}product kept {products_kept(old, case)!s:6}"
                  f"new numbers {new_numbers(old, case['turns'])!s:10}{old!r}")

⚠️ Set OLD_REWRITE, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: Function must have a docstring if description not provided.` | `@tool` on a function with no docstring | Add a docstring, or pass `description=` as Part D does for `search_named`; the model reads it |
| `IndexError: list index out of range` at `reply.tool_calls[0]` | the model answered in text and called no tool (it does this for a turn such as *Thanks, that is all.*) | Read `reply.tool_calls` before you index it, as `tool_query()` and `tool_answer()` do |
| `OpenAIInvalidRequestError: Error code: 400 ... An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'.` | the model's tool call was sent back without a `ToolMessage` for each call | Append a result for every call before the next `invoke()`, as `tool_answer()` does |
| `OpenAIInvalidRequestError: Error code: 400 ... Function tools with reasoning_effort are not supported for gpt-6-luna in /v1/chat/completions.` | `REASONING_EFFORT` is `low`, `medium` or `high` for a reasoning model | Set `REASONING_EFFORT=none` in `.env`: Chat Completions takes tools on `gpt-6-luna` only at that effort |

## Resources

- [LangChain: retrieval](https://docs.langchain.com/oss/python/langchain/retrieval): 2-step RAG, agentic RAG with
  retrieval as a tool, and query rewriting as a step between them
- [LangChain: tools](https://docs.langchain.com/oss/python/langchain/tools): `@tool`, descriptions and argument
  schemas
- [LangChain: models, tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling): `bind_tools`,
  `tool_calls`, the execution loop and `ToolMessage`
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): what the API receives
  for a tool and returns for a call

Next: [Lab 10](../10_Long_Context_Or_Retrieval/README.md), *How to choose between long context and retrieval*. Back to [the lab list](../README.md).